# import

In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import paramiko
from scp import SCPClient
import os
import pandas as pd
import numpy as np
import re
import time
import subprocess
import ast
from datetime import datetime
import shutil
from datetime import datetime

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from sklearn.model_selection import train_test_split,KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,make_scorer
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.data import Data, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.metrics import log_loss
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import gc
import shutil
import posixpath
from scipy.stats import pearsonr, spearmanr, mannwhitneyu, ttest_ind, ks_2samp
import inspect
import glob
from sklearn.linear_model import Ridge
from rdkit import DataStructs

# 各モデルの成功MD＋nonMD

In [6]:
import os
import pandas as pd

save_dir = "/Users/teraimao/experiment/data/解析結果"

NONMD_CSV = "/Users/teraimao/experiment/data/解析結果/スライド検証用/baselineB_dataset.csv"

md_files = {
    "gbsw": "gbsw_common_successID.csv",
    "hdgb": "hdgb_common_successID.csv",
    "hdgbvdw": "hdgbvdw_common_successID.csv",
}

output_files = {
    "gbsw": "gbsw_common_successID_baselineB.csv",
    "hdgb": "hdgb_common_successID_baselineB.csv",
    "hdgbvdw": "hdgbvdw_common_successID_baselineB.csv",
}

id_col = "ID"
target_col = "Permeability"

# =========================
# baselineB 読み込み
# =========================

nonmd = pd.read_csv(NONMD_CSV)

nonmd[id_col] = nonmd[id_col].astype(int)

print("===== baselineB =====")
print("path:", NONMD_CSV)
print("shape:", nonmd.shape)
print("unique IDs:", nonmd[id_col].nunique())
print("duplicate IDs:", nonmd.duplicated(id_col).sum())
print("target NaN:", nonmd[target_col].isna().sum())

# 念のためID重複があった場合だけ除去
if nonmd.duplicated(id_col).sum() > 0:
    nonmd = nonmd.drop_duplicates(subset=[id_col], keep="first").copy()

# =========================
# MD + baselineB 結合
# =========================

for model, md_fname in md_files.items():
    print("\n" + "="*80)
    print("model:", model)

    md_path = os.path.join(save_dir, md_fname)
    out_path = os.path.join(save_dir, output_files[model])

    md = pd.read_csv(md_path)

    md[id_col] = md[id_col].astype(int)
    md["Pos_Num"] = md["Pos_Num"].astype(int)

    print("MD path:", md_path)
    print("MD rows:", len(md))
    print("MD unique IDs:", md[id_col].nunique())
    print("MD Status counts:")
    print(md["Status"].value_counts(dropna=False))

    merged = md.merge(
        nonmd,
        on=id_col,
        how="inner",
        suffixes=("", "_baselineB")
    )

    print("merged rows:", len(merged))
    print("merged unique IDs:", merged[id_col].nunique())
    print("target NaN:", merged[target_col].isna().sum())
    print("duplicate ID/Pos:", merged.duplicated([id_col, "Pos_Num"]).sum())

    pos_count = merged.groupby(id_col)["Pos_Num"].nunique()
    print("pos count distribution:")
    print(pos_count.value_counts().sort_index())

    missing_ids = sorted(set(md[id_col].unique()) - set(merged[id_col].unique()))
    print("missing IDs from baselineB:", len(missing_ids))
    if len(missing_ids) > 0:
        print("missing ID examples:", missing_ids[:20])

    merged.to_csv(out_path, index=False)
    print("saved:", out_path)

print("\n✅ MD + baselineB merged CSVs created")

===== baselineB =====
path: /Users/teraimao/experiment/data/解析結果/スライド検証用/baselineB_dataset.csv
shape: (8466, 142)
unique IDs: 8466
duplicate IDs: 0
target NaN: 0

model: gbsw
MD path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID.csv
MD rows: 37630
MD unique IDs: 7526
MD Status counts:
Status
Success    37630
Name: count, dtype: int64
merged rows: 37630
merged unique IDs: 7526
target NaN: 0
duplicate ID/Pos: 0
pos count distribution:
Pos_Num
5    7526
Name: count, dtype: int64
missing IDs from baselineB: 0
saved: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB.csv

model: hdgb
MD path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID.csv
MD rows: 37630
MD unique IDs: 7526
MD Status counts:
Status
Success    37630
Name: count, dtype: int64
merged rows: 37630
merged unique IDs: 7526
target NaN: 0
duplicate ID/Pos: 0
pos count distribution:
Pos_Num
5    7526
Name: count, dtype: int64
missing IDs from baselineB: 0
saved: /Users/teraimao/experimen

In [7]:
import os
import pandas as pd

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "gbsw": "gbsw_common_successID_baselineB.csv",
    "hdgb": "hdgb_common_successID_baselineB.csv",
    "hdgbvdw": "hdgbvdw_common_successID_baselineB.csv",
}

id_col = "ID"
pos_col = "Pos_Num"

md_feature_cols = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

# IDごとに1つだけ残すnonMD列
# ここに入っていないMD由来の管理列・パス列・状態列は基本的に落とす
md_metadata_cols = [
    "Model",
    "Date",
    "t_end_ps",
    "Cons_Label",
    "Pos_Num",
    "Z",
    "WorkDir",
    "LogPath",
    "HB_Path",
    "SASA_Path",
    "Status",
    "Message",
    "skip_ps",
    "dt_ps",
    "tolerance_frames",
    "Normal_Termination",
] + md_feature_cols

for model, fname in files.items():
    print("\n" + "="*80)
    print("model:", model)

    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path)

    print("input path:", path)
    print("input rows:", len(df))
    print("input unique IDs:", df[id_col].nunique())

    df[id_col] = df[id_col].astype(int)
    df[pos_col] = df[pos_col].astype(int)

    # 念のため確認
    pos_count = df.groupby(id_col)[pos_col].nunique()
    print("pos count distribution before:")
    print(pos_count.value_counts().sort_index())

    if not (pos_count == 5).all():
        raise ValueError(f"{model}: pos 1〜5 がそろっていないIDがあります")

    if df.duplicated([id_col, pos_col]).sum() > 0:
        raise ValueError(f"{model}: duplicate ID/Pos があります")

    # =========================
    # 1. nonMD部分をIDごとに1行だけ残す
    # =========================

    nonmd_cols = [c for c in df.columns if c not in md_metadata_cols]

    # IDは必ず入れる
    if id_col not in nonmd_cols:
        nonmd_cols = [id_col] + nonmd_cols

    nonmd = df[nonmd_cols].drop_duplicates(subset=[id_col], keep="first").copy()

    print("nonMD columns:", len(nonmd.columns))
    print("nonMD rows:", len(nonmd))

    # =========================
    # 2. MD特徴量をposごとに横展開
    # =========================

    md = df[[id_col, pos_col] + md_feature_cols].copy()

    md_wide = md.pivot(
        index=id_col,
        columns=pos_col,
        values=md_feature_cols
    )

    # 列名を HB_Avg_pos1 の形にする
    md_wide.columns = [
        f"{feature}_pos{pos}"
        for feature, pos in md_wide.columns
    ]

    md_wide = md_wide.reset_index()

    print("MD wide shape:", md_wide.shape)

    # =========================
    # 3. nonMD + 横展開MD を結合
    # =========================

    wide = nonmd.merge(md_wide, on=id_col, how="inner")

    wide = wide.sort_values(id_col).reset_index(drop=True)

    print("output rows:", len(wide))
    print("output unique IDs:", wide[id_col].nunique())
    print("output columns:", len(wide.columns))

    # 確認
    if "Permeability" in wide.columns:
        print("Permeability NaN:", wide["Permeability"].isna().sum())

    md_wide_cols = [c for c in wide.columns if any(c.startswith(f"{m}_pos") for m in md_feature_cols)]
    print("MD wide feature columns:", len(md_wide_cols))
    print("MD wide NaN total:", wide[md_wide_cols].isna().sum().sum())

    assert len(wide) == df[id_col].nunique()
    assert wide[id_col].duplicated().sum() == 0
    assert wide[md_wide_cols].isna().sum().sum() == 0

    # =========================
    # 4. 既存ファイルを消して、同じ名前で保存
    # =========================

    if os.path.exists(path):
        os.remove(path)
        print("deleted old file:", path)

    wide.to_csv(path, index=False)
    print("saved new wide file:", path)

print("\n✅ All files were replaced with ID-level wide CSVs")


model: gbsw
input path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB.csv
input rows: 37630
input unique IDs: 7526
pos count distribution before:
Pos_Num
5    7526
Name: count, dtype: int64
nonMD columns: 142
nonMD rows: 7526
MD wide shape: (7526, 26)
output rows: 7526
output unique IDs: 7526
output columns: 167
Permeability NaN: 0
MD wide feature columns: 25
MD wide NaN total: 0
deleted old file: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB.csv
saved new wide file: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB.csv

model: hdgb
input path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB.csv
input rows: 37630
input unique IDs: 7526
pos count distribution before:
Pos_Num
5    7526
Name: count, dtype: int64
nonMD columns: 142
nonMD rows: 7526
MD wide shape: (7526, 26)
output rows: 7526
output unique IDs: 7526
output columns: 167
Permeability NaN: 0
MD wide feature columns: 25
MD wide NaN total: 0
d

# （PAMPA -10などの制限なし）３モデルMDのみ比較

In [9]:
# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "GBSW": "gbsw_common_successID_baselineB.csv",
    "HDGB": "hdgb_common_successID_baselineB.csv",
    "HDGBvdW": "hdgbvdw_common_successID_baselineB.csv",
}

id_col = "ID"
target_col = "Permeability"

base_md_features = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

feature_cols = [
    f"{feat}_pos{pos}"
    for feat in base_md_features
    for pos in [1, 2, 3, 4, 5]
]

random_state = 42
test_size = 0.2

# =========================
# 評価関数：Pearson修正版
# =========================

def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    pearson = np.corrcoef(y_true, y_pred)[0, 1]
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "Pearson_R": pearson,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
    }

# =========================
# データ読み込み
# =========================

datasets = {}

for model_name, fname in files.items():
    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path)

    df[id_col] = df[id_col].astype(int)

    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{model_name}: MD特徴量が足りません: {missing}")

    if target_col not in df.columns:
        raise ValueError(f"{model_name}: {target_col} がありません")

    use_df = df[[id_col, target_col] + feature_cols].copy()

    use_df[target_col] = pd.to_numeric(use_df[target_col], errors="coerce")
    for c in feature_cols:
        use_df[c] = pd.to_numeric(use_df[c], errors="coerce")

    before = len(use_df)
    use_df = use_df.dropna(subset=[target_col] + feature_cols).copy()
    after = len(use_df)

    print("\n" + "="*80)
    print(model_name)
    print("path:", path)
    print("rows before:", before)
    print("rows used  :", after)
    print("unique IDs :", use_df[id_col].nunique())
    print("features   :", len(feature_cols))
    print("target min/max/mean/std:")
    print(use_df[target_col].describe())

    datasets[model_name] = use_df

# =========================
# 3モデル共通IDにそろえる
# =========================

common_ids = None

for model_name, df in datasets.items():
    ids = set(df[id_col].unique())
    common_ids = ids if common_ids is None else common_ids & ids

common_ids = sorted(common_ids)

print("\n" + "="*80)
print("common IDs:", len(common_ids))

for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(common_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)
    print(model_name, "rows after common ID filter:", len(datasets[model_name]))

# =========================
# 同一 train/test split
# =========================

train_ids, test_ids = train_test_split(
    common_ids,
    test_size=test_size,
    random_state=random_state
)

train_ids = set(train_ids)
test_ids = set(test_ids)

print("\ntrain IDs:", len(train_ids))
print("test IDs :", len(test_ids))

# =========================
# Random Forestで比較
# =========================

results = []
predictions = []
feature_importances = []

for model_name, df in datasets.items():
    print("\n" + "="*80)
    print("Training:", model_name)

    train_df = df[df[id_col].isin(train_ids)].copy()
    test_df = df[df[id_col].isin(test_ids)].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]

    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=random_state,
        n_jobs=-1,
        min_samples_leaf=3
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    metrics = calc_metrics(y_test, y_pred)

    results.append({
        "MD_model": model_name,
        "N_total": len(df),
        "N_train": len(train_df),
        "N_test": len(test_df),
        "N_features": len(feature_cols),
        **metrics,
    })

    print(f"Pearson R: {metrics['Pearson_R']:.4f}")
    print(f"R2       : {metrics['R2']:.4f}")
    print(f"MAE      : {metrics['MAE']:.4f}")
    print(f"RMSE     : {metrics['RMSE']:.4f}")

    predictions.append(pd.DataFrame({
        id_col: test_df[id_col].values,
        "MD_model": model_name,
        "y_true": np.asarray(y_test),
        "y_pred": y_pred,
    }))

    feature_importances.append(pd.DataFrame({
        "MD_model": model_name,
        "Feature": feature_cols,
        "Importance": rf.feature_importances_,
    }).sort_values("Importance", ascending=False))

# =========================
# 保存
# =========================

results_df = pd.DataFrame(results)
predictions_df = pd.concat(predictions, ignore_index=True)
importance_df = pd.concat(feature_importances, ignore_index=True)

metrics_out = os.path.join(save_dir, "rf_3model_MD25_only_metrics_fixedPearson.csv")
pred_out = os.path.join(save_dir, "rf_3model_MD25_only_predictions_fixedPearson.csv")
imp_out = os.path.join(save_dir, "rf_3model_MD25_only_feature_importance_fixedPearson.csv")

results_df.to_csv(metrics_out, index=False)
predictions_df.to_csv(pred_out, index=False)
importance_df.to_csv(imp_out, index=False)

print("\n" + "="*80)
print("Saved:")
print(metrics_out)
print(pred_out)
print(imp_out)

print("\n===== 3 model comparison: MD25 only =====")
display(results_df.sort_values("R2", ascending=False))

print("\n===== Top 10 feature importance per MD model =====")
display(importance_df.groupby("MD_model").head(10))


GBSW
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB.csv
rows before: 7526
rows used  : 7526
unique IDs : 7526
features   : 25
target min/max/mean/std:
count    7526.000000
mean       -5.947189
std         1.083099
min       -10.000000
25%        -6.330000
50%        -5.740000
75%        -5.290000
max        -3.900000
Name: Permeability, dtype: float64

HDGB
path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB.csv
rows before: 7526
rows used  : 7526
unique IDs : 7526
features   : 25
target min/max/mean/std:
count    7526.000000
mean       -5.947189
std         1.083099
min       -10.000000
25%        -6.330000
50%        -5.740000
75%        -5.290000
max        -3.900000
Name: Permeability, dtype: float64

HDGBvdW
path: /Users/teraimao/experiment/data/解析結果/hdgbvdw_common_successID_baselineB.csv
rows before: 7526
rows used  : 7526
unique IDs : 7526
features   : 25
target min/max/mean/std:
count    7526.000000
mean       -5.947189
std   

,MD_model,N_total,N_train,N_test,N_features,Pearson_R,R2,MAE,RMSE
0,GBSW,7526,6020,1506,25,0.381047,0.140545,0.693532,1.013991
2,HDGBvdW,7526,6020,1506,25,0.378784,0.137959,0.694613,1.015515
1,HDGB,7526,6020,1506,25,0.366185,0.128015,0.705032,1.021355



===== Top 10 feature importance per MD model =====


,MD_model,Feature,Importance
0,GBSW,HB_Avg_pos1,0.049773
1,GBSW,SASA_Avg_pos1,0.045543
2,GBSW,vdW_Avg_pos5,0.045160
3,GBSW,HB_Avg_pos2,0.045073
4,GBSW,Coul_Avg_pos3,0.044610
5,GBSW,vdW_Avg_pos1,0.044333
6,GBSW,Solv_Free_Avg_pos2,0.043312
7,GBSW,Coul_Avg_pos1,0.043225
8,GBSW,vdW_Avg_pos2,0.042325
9,GBSW,vdW_Avg_pos3,0.041795


# PAMPA,-10などを除いたcsv

In [10]:
save_dir = "/Users/teraimao/experiment/data/解析結果"

DB_CSV = "/Users/teraimao/experiment/data/all_csv_data/strict_this_is_CycPeptMPDB_PAMPA.csv"

input_files = {
    "gbsw": "gbsw_common_successID_baselineB.csv",
    "hdgb": "hdgb_common_successID_baselineB.csv",
    "hdgbvdw": "hdgbvdw_common_successID_baselineB.csv",
}

output_files = {
    "gbsw": "gbsw_common_successID_baselineB_strictPAMPA.csv",
    "hdgb": "hdgb_common_successID_baselineB_strictPAMPA.csv",
    "hdgbvdw": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
}

id_col = "ID"
strict_target_col = "PAMPA"
new_target_col = "PAMPA_logPm"

md25_cols = [
    f"{feat}_pos{pos}"
    for feat in ["HB_Avg", "SASA_Avg", "Solv_Free_Avg", "vdW_Avg", "Coul_Avg"]
    for pos in [1, 2, 3, 4, 5]
]

def is_blank_value(x):
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in ["nan", "none", "null", "na", "n/a", "<na>"]

# =========================
# 1. strict PAMPA target作成
# =========================

db = pd.read_csv(DB_CSV, low_memory=False)

print("===== strict PAMPA DB =====")
print("path:", DB_CSV)
print("shape:", db.shape)
print("columns:", db.columns.tolist())

if id_col not in db.columns:
    raise ValueError("DBにID列がありません")

if strict_target_col not in db.columns:
    raise ValueError("DBにPAMPA列がありません")

target = db.copy()

target[id_col] = pd.to_numeric(target[id_col], errors="coerce")
target[strict_target_col] = pd.to_numeric(target[strict_target_col], errors="coerce")

before = len(target)
target = target.dropna(subset=[id_col, strict_target_col]).copy()
target[id_col] = target[id_col].astype(int)
print("\nafter dropna ID/PAMPA:", len(target), "removed:", before - len(target))

# Detection Limitが残っていたら除外
for dl_col in ["Detection_Limit_1", "Detection_Limit_2"]:
    if dl_col in target.columns:
        before = len(target)
        target = target[target[dl_col].apply(is_blank_value)].copy()
        print(f"after remove {dl_col} not blank:", len(target), "removed:", before - len(target))
    else:
        print(f"{dl_col}: column not found")

# PAMPA=-10を除外
before = len(target)
target = target[target[strict_target_col] != -10].copy()
print("after remove PAMPA=-10:", len(target), "removed:", before - len(target))

# ID重複除去
before = len(target)
target = target[[id_col, strict_target_col]].drop_duplicates(subset=[id_col], keep="first").copy()
target = target.rename(columns={strict_target_col: new_target_col})
print("after drop duplicate IDs:", len(target), "removed:", before - len(target))

print("\nstrict target summary:")
print("unique IDs:", target[id_col].nunique())
print("target NaN:", target[new_target_col].isna().sum())
print(target[new_target_col].describe())

# =========================
# 2. 各MD wide CSVにstrict PAMPAを結合
# =========================

target_like_exact_cols = [
    "Permeability",
    "permeability",
    "PAMPA",
    "PAMPA_logPm",
    "logPm",
    "LogPm",
    "logPapp",
    "LogPapp",
    "PAMPA_logPapp",
]

for model, fname in input_files.items():
    print("\n" + "="*80)
    print("model:", model)

    in_path = os.path.join(save_dir, fname)
    out_path = os.path.join(save_dir, output_files[model])

    df = pd.read_csv(in_path, low_memory=False)
    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df = df.dropna(subset=[id_col]).copy()
    df[id_col] = df[id_col].astype(int)

    print("input:", in_path)
    print("input shape:", df.shape)
    print("input unique IDs:", df[id_col].nunique())
    print("input duplicate IDs:", df[id_col].duplicated().sum())

    # 念のため、今入っているPermeabilityとstrict PAMPAの違いを確認
    if "Permeability" in df.columns:
        old_y = df[[id_col, "Permeability"]].drop_duplicates(subset=[id_col], keep="first").copy()
        old_y["Permeability"] = pd.to_numeric(old_y["Permeability"], errors="coerce")
        check = old_y.merge(target, on=id_col, how="inner")

        print("\nold Permeability vs strict PAMPA:")
        print("common IDs for check:", len(check))
        if len(check) > 0:
            print("corr:", check["Permeability"].corr(check[new_target_col]))
            print("mean old Permeability:", check["Permeability"].mean())
            print("mean strict PAMPA    :", check[new_target_col].mean())
            print("max abs diff:", (check["Permeability"] - check[new_target_col]).abs().max())

    # 古い目的変数っぽい列は落として、strict PAMPAだけを目的変数にする
    drop_cols = [c for c in target_like_exact_cols if c in df.columns and c != id_col]
    print("\ndrop old target-like columns:", drop_cols)

    df_no_target = df.drop(columns=drop_cols).copy()

    # strict PAMPAを結合
    merged = target.merge(df_no_target, on=id_col, how="inner")

    # 列順を整える
    other_cols = [c for c in merged.columns if c not in [id_col, new_target_col]]
    merged = merged[[id_col, new_target_col] + other_cols].copy()

    print("\nmerged shape:", merged.shape)
    print("merged unique IDs:", merged[id_col].nunique())
    print("target NaN:", merged[new_target_col].isna().sum())
    print("duplicate IDs:", merged[id_col].duplicated().sum())

    missing_md = [c for c in md25_cols if c not in merged.columns]
    if missing_md:
        raise ValueError(f"{model}: MD25列が足りません: {missing_md}")

    print("MD25 NaN total:", merged[md25_cols].isna().sum().sum())

    # 保存
    merged.to_csv(out_path, index=False)
    print("saved:", out_path)

print("\n✅ strict PAMPA CSVs created")

===== strict PAMPA DB =====
path: /Users/teraimao/experiment/data/all_csv_data/strict_this_is_CycPeptMPDB_PAMPA.csv
shape: (6925, 265)
columns: ['ID', 'Source', 'Year', 'Version', 'Original_Name_in_Source_Literature', 'Structurally_Unique_ID', 'Same_Peptides_ID', 'Same_Peptides_Source', 'Same_Peptides_Permeability', 'Same_Peptides_Assay', 'SMILES', 'HELM', 'N-number', 'C-number', 'HELM_URL', 'Sequence', 'Modifier', 'Patch_From_Modifier', 'Clean_Sequence', 'Cycle_Length', 'newN_number', 'newC_number', 'Modifier_Warning', 'Cyc_Patch', 'Sequence_LogP', 'Sequence_TPSA', 'Monomer_Length', 'Monomer_Length_in_Main_Chain', 'Molecule_Shape', 'Permeability', 'PAMPA', 'Caco2', 'MDCK', 'RRCK', 'Detection_Limit_1', 'Detection_Limit_2', 'R_PAMAP', 'R_Caco2', 'R_MDCK', 'R_RRCK', 'T_PAMPA', 'EPSA', 'PSA', '_3DPSA', 'NULL', 'MaxEStateIndex', 'MinEStateIndex', 'MaxAbsEStateIndex', 'MinAbsEStateIndex', 'qed', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'NumValenceElectrons', 'NumRadicalElectrons', 'MaxParti

# 予測条件

# 3つのモデル比較 MDのみ

In [12]:
# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "GBSW": "gbsw_common_successID_baselineB_strictPAMPA.csv",
    "HDGB": "hdgb_common_successID_baselineB_strictPAMPA.csv",
    "HDGBvdW": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
}

id_col = "ID"
target_col = "PAMPA_logPm"

base_md_features = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

feature_cols = [
    f"{feat}_pos{pos}"
    for feat in base_md_features
    for pos in [1, 2, 3, 4, 5]
]

RANDOM_STATE = 42
TEST_SIZE = 0.2

# 全strict PAMPA共通IDを使う
MAX_ID = None

# 前回のID<=1000条件に寄せたい場合だけこちらに変更
# MAX_ID = 1000

# =========================
# データ読み込み
# =========================

datasets = {}

for model_name, fname in files.items():
    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path, low_memory=False)

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    df = df.dropna(subset=[id_col, target_col]).copy()
    df[id_col] = df[id_col].astype(int)

    if MAX_ID is not None:
        df = df[df[id_col] <= MAX_ID].copy()

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{model_name}: 足りないMD特徴量があります: {missing_cols}")

    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    before = len(df)
    df = df.dropna(subset=feature_cols).copy()
    after = len(df)

    df = df.drop_duplicates(subset=[id_col], keep="first").copy()

    print("\n" + "="*80)
    print(model_name)
    print("path:", path)
    print("rows before dropna:", before)
    print("rows used:", after)
    print("unique IDs:", df[id_col].nunique())
    print("target describe:")
    print(df[target_col].describe())

    datasets[model_name] = df[[id_col, target_col] + feature_cols].copy()

# =========================
# 3モデル共通IDにそろえる
# =========================

common_ids = None

for model_name, df in datasets.items():
    ids = set(df[id_col].unique())
    common_ids = ids if common_ids is None else common_ids & ids

common_ids = sorted(common_ids)

print("\n" + "="*80)
print("common IDs:", len(common_ids))

for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(common_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)

    print(model_name, "rows after common ID filter:", len(datasets[model_name]))

# =========================
# 同一train/test split
# =========================

indices = np.arange(len(common_ids))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\n===== split =====")
print("n train:", len(train_idx))
print("n test :", len(test_idx))

# =========================
# Random Forest Regressorで評価
# =========================

results = []
predictions = []
importances = []

for model_name, df in datasets.items():
    print("\n" + "="*80)
    print("Training:", model_name)

    X = df[feature_cols].copy()
    y = df[target_col].copy()
    ids = df[id_col].copy()

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    id_test = ids.iloc[test_idx]

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2,
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    r, p = pearsonr(np.asarray(y_test), np.asarray(y_pred))
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print("n features:", X.shape[1])
    print("n train   :", len(y_train))
    print("n test    :", len(y_test))
    print("Pearson R :", round(r, 4))
    print("R2        :", round(r2, 4))
    print("MAE       :", round(mae, 4))
    print("RMSE      :", round(rmse, 4))

    results.append({
        "MD_model": model_name,
        "Dataset": "strict_PAMPA",
        "ID_filter": f"ID<={MAX_ID}" if MAX_ID is not None else "all_common_ID",
        "N_total_common_ID": len(df),
        "N_train": len(y_train),
        "N_test": len(y_test),
        "N_features": X.shape[1],
        "R": r,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
    })

    pred_df = pd.DataFrame({
        "ID": id_test.values,
        "MD_model": model_name,
        "y_true_PAMPA_logPm": y_test.values,
        "y_pred_PAMPA_logPm": y_pred,
        "error": y_pred - y_test.values,
        "abs_error": np.abs(y_pred - y_test.values),
    })
    predictions.append(pred_df)

    imp_df = pd.DataFrame({
        "MD_model": model_name,
        "Feature": feature_cols,
        "Importance": rf.feature_importances_,
    }).sort_values("Importance", ascending=False)
    importances.append(imp_df)

# =========================
# 保存
# =========================

results_df = pd.DataFrame(results)
predictions_df = pd.concat(predictions, ignore_index=True)
importances_df = pd.concat(importances, ignore_index=True)

suffix = "ID1000" if MAX_ID is not None else "allIDs"

metrics_out = os.path.join(save_dir, f"rf_3model_MD25_only_strictPAMPA_{suffix}_metrics.csv")
pred_out = os.path.join(save_dir, f"rf_3model_MD25_only_strictPAMPA_{suffix}_predictions.csv")
imp_out = os.path.join(save_dir, f"rf_3model_MD25_only_strictPAMPA_{suffix}_feature_importance.csv")

results_df.to_csv(metrics_out, index=False)
predictions_df.to_csv(pred_out, index=False)
importances_df.to_csv(imp_out, index=False)

print("\n" + "="*80)
print("Saved:")
print(metrics_out)
print(pred_out)
print(imp_out)

print("\n===== 3 model comparison: MD25 only / strict PAMPA =====")
display(results_df.sort_values("R2", ascending=False))

print("\n===== Top 10 feature importance per model =====")
display(importances_df.groupby("MD_model").head(10))


GBSW
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA.csv
rows before dropna: 6441
rows used: 6441
unique IDs: 6441
target describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
max        -3.900000
Name: PAMPA_logPm, dtype: float64

HDGB
path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB_strictPAMPA.csv
rows before dropna: 6441
rows used: 6441
unique IDs: 6441
target describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
max        -3.900000
Name: PAMPA_logPm, dtype: float64

HDGBvdW
path: /Users/teraimao/experiment/data/解析結果/hdgbvdw_common_successID_baselineB_strictPAMPA.csv
rows before dropna: 6441
rows used: 6441
unique IDs: 6441
target describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        

,MD_model,Dataset,ID_filter,N_total_common_ID,N_train,N_test,N_features,R,R2,MAE,RMSE
2,HDGBvdW,strict_PAMPA,all_common_ID,6441,5152,1289,25,0.667227,0.432938,0.441581,0.574912
0,GBSW,strict_PAMPA,all_common_ID,6441,5152,1289,25,0.664154,0.427966,0.445897,0.577426
1,HDGB,strict_PAMPA,all_common_ID,6441,5152,1289,25,0.649061,0.409235,0.450879,0.586804



===== Top 10 feature importance per model =====


,MD_model,Feature,Importance
0,GBSW,vdW_Avg_pos1,0.054129
1,GBSW,SASA_Avg_pos3,0.053159
2,GBSW,SASA_Avg_pos5,0.051392
3,GBSW,vdW_Avg_pos2,0.050782
4,GBSW,vdW_Avg_pos5,0.050071
5,GBSW,vdW_Avg_pos3,0.048882
6,GBSW,SASA_Avg_pos4,0.048789
7,GBSW,vdW_Avg_pos4,0.047940
8,GBSW,SASA_Avg_pos1,0.047786
9,GBSW,SASA_Avg_pos2,0.047630


# フィンガープリントのみ

## 使用Morgan fingerprint

## 予測

In [15]:

# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

# 3つとも同じID・同じPAMPA_logPmなので、代表としてGBSW版を使う
INPUT_CSV = os.path.join(save_dir, "gbsw_common_successID_baselineB_strictPAMPA.csv")

id_col = "ID"
smiles_col = "SMILES"
target_col = "PAMPA_logPm"

RANDOM_STATE = 42
TEST_SIZE = 0.2

# Morgan fingerprint設定
FP_RADIUS = 2
FP_BITS = 2048

# =========================
# fingerprint作成関数
# =========================

def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

# =========================
# データ読み込み
# =========================

df = pd.read_csv(INPUT_CSV, low_memory=False)

print("===== input =====")
print("path:", INPUT_CSV)
print("shape:", df.shape)
print("unique IDs:", df[id_col].nunique())

required_cols = [id_col, smiles_col, target_col]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"必要な列がありません: {missing}")

df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

df = df.dropna(subset=[id_col, smiles_col, target_col]).copy()
df[id_col] = df[id_col].astype(int)

# ID重複なしのはずだが念のため
df = df.drop_duplicates(subset=[id_col], keep="first").copy()
df = df.sort_values(id_col).reset_index(drop=True)

print("\n===== after cleaning =====")
print("rows:", len(df))
print("unique IDs:", df[id_col].nunique())
print("target describe:")
print(df[target_col].describe())

# =========================
# fingerprint作成
# =========================

fps = []
valid_indices = []

for i, smi in enumerate(df[smiles_col]):
    arr = smiles_to_morgan_fp(smi, radius=FP_RADIUS, n_bits=FP_BITS)

    if arr is not None:
        fps.append(arr)
        valid_indices.append(i)

print("\n===== fingerprint =====")
print("valid molecules:", len(valid_indices))
print("invalid molecules:", len(df) - len(valid_indices))

df_valid = df.iloc[valid_indices].copy().reset_index(drop=True)
X = np.vstack(fps)
y = df_valid[target_col].values
ids = df_valid[id_col].values

print("X shape:", X.shape)
print("y shape:", y.shape)

# =========================
# train/test split
# =========================

indices = np.arange(len(df_valid))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]
id_test = ids[test_idx]

print("\n===== split =====")
print("n train:", len(train_idx))
print("n test :", len(test_idx))

# =========================
# Random Forest Regressor
# =========================

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2,
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r, p = pearsonr(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n" + "="*80)
print("Morgan fingerprint only")
print("n samples :", len(df_valid))
print("n features:", X.shape[1])
print("n train   :", len(y_train))
print("n test    :", len(y_test))
print("Pearson R :", round(r, 4))
print("R2        :", round(r2, 4))
print("MAE       :", round(mae, 4))
print("RMSE      :", round(rmse, 4))

# =========================
# 保存
# =========================

metrics_df = pd.DataFrame([{
    "Feature_set": "Morgan fingerprint only",
    "Dataset": "strict_PAMPA",
    "N_total_common_ID": len(df_valid),
    "N_train": len(y_train),
    "N_test": len(y_test),
    "N_features": X.shape[1],
    "FP_radius": FP_RADIUS,
    "FP_bits": FP_BITS,
    "R": r,
    "R2": r2,
    "MAE": mae,
    "RMSE": rmse,
}])

pred_df = pd.DataFrame({
    "ID": id_test,
    "y_true_PAMPA_logPm": y_test,
    "y_pred_PAMPA_logPm": y_pred,
    "error": y_pred - y_test,
    "abs_error": np.abs(y_pred - y_test),
})

metrics_out = os.path.join(save_dir, "rf_fingerprint_only_strictPAMPA_metrics.csv")
pred_out = os.path.join(save_dir, "rf_fingerprint_only_strictPAMPA_predictions.csv")

metrics_df.to_csv(metrics_out, index=False)
pred_df.to_csv(pred_out, index=False)

print("\n===== saved =====")
print(metrics_out)
print(pred_out)

display(metrics_df)

===== input =====
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA.csv
shape: (6441, 167)
unique IDs: 6441

===== after cleaning =====
rows: 6441
unique IDs: 6441
target describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
max        -3.900000
Name: PAMPA_logPm, dtype: float64

===== fingerprint =====
valid molecules: 6441
invalid molecules: 0


[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerator
[16:30:56] DEPRECATION WARNING: please use MorganGenerat

X shape: (6441, 2048)
y shape: (6441,)

===== split =====
n train: 5152
n test : 1289

Morgan fingerprint only
n samples : 6441
n features: 2048
n train   : 5152
n test    : 1289
Pearson R : 0.7858
R2        : 0.6048
MAE       : 0.3683
RMSE      : 0.4799

===== saved =====
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_only_strictPAMPA_metrics.csv
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_only_strictPAMPA_predictions.csv


,Feature_set,Dataset,N_total_common_ID,N_train,N_test,N_features,FP_radius,FP_bits,R,R2,MAE,RMSE
0,Morgan fingerprint only,strict_PAMPA,6441,5152,1289,2048,2,2048,0.785792,0.604845,0.368293,0.479921


# Fingerprint + MD特徴量

In [16]:

# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "GBSW": "gbsw_common_successID_baselineB_strictPAMPA.csv",
    "HDGB": "hdgb_common_successID_baselineB_strictPAMPA.csv",
    "HDGBvdW": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
}

id_col = "ID"
smiles_col = "SMILES"
target_col = "PAMPA_logPm"

base_md_features = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

md25_cols = [
    f"{feat}_pos{pos}"
    for feat in base_md_features
    for pos in [1, 2, 3, 4, 5]
]

RANDOM_STATE = 42
TEST_SIZE = 0.2

FP_RADIUS = 2
FP_BITS = 2048

# =========================
# Fingerprint作成関数
# =========================

def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr


def calc_metrics(y_true, y_pred):
    r, p = pearsonr(np.asarray(y_true), np.asarray(y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "R": r,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
    }


def evaluate_rf(X, y, ids, train_idx, test_idx, feature_set_name):
    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]
    id_test = ids[test_idx]

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2,
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    metrics = calc_metrics(y_test, y_pred)

    print("\n" + "="*80)
    print(feature_set_name)
    print("n features:", X.shape[1])
    print("n train   :", len(y_train))
    print("n test    :", len(y_test))
    print("Pearson R :", round(metrics["R"], 4))
    print("R2        :", round(metrics["R2"], 4))
    print("MAE       :", round(metrics["MAE"], 4))
    print("RMSE      :", round(metrics["RMSE"], 4))

    result = {
        "Feature_set": feature_set_name,
        "Dataset": "strict_PAMPA",
        "N_total_common_ID": len(y),
        "N_train": len(y_train),
        "N_test": len(y_test),
        "N_features": X.shape[1],
        "FP_radius": FP_RADIUS if "Fingerprint" in feature_set_name else "",
        "FP_bits": FP_BITS if "Fingerprint" in feature_set_name else "",
        **metrics,
    }

    pred_df = pd.DataFrame({
        "ID": id_test,
        "Feature_set": feature_set_name,
        "y_true_PAMPA_logPm": y_test,
        "y_pred_PAMPA_logPm": y_pred,
        "error": y_pred - y_test,
        "abs_error": np.abs(y_pred - y_test),
    })

    return result, pred_df, rf


# =========================
# 1. 各CSV読み込み
# =========================

datasets = {}

for model_name, fname in files.items():
    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path, low_memory=False)

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    df = df.dropna(subset=[id_col, target_col, smiles_col]).copy()
    df[id_col] = df[id_col].astype(int)
    df = df.drop_duplicates(subset=[id_col], keep="first").copy()

    missing_md = [c for c in md25_cols if c not in df.columns]
    if missing_md:
        raise ValueError(f"{model_name}: MD25列が足りません: {missing_md}")

    for c in md25_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=md25_cols).copy()
    df = df.sort_values(id_col).reset_index(drop=True)

    print("\n" + "="*80)
    print(model_name)
    print("path:", path)
    print("rows:", len(df))
    print("unique IDs:", df[id_col].nunique())
    print("PAMPA_logPm describe:")
    print(df[target_col].describe())

    datasets[model_name] = df[[id_col, smiles_col, target_col] + md25_cols].copy()

# =========================
# 2. 3モデル共通IDにそろえる
# =========================

common_ids = None

for model_name, df in datasets.items():
    ids = set(df[id_col].unique())
    common_ids = ids if common_ids is None else common_ids & ids

common_ids = sorted(common_ids)

print("\n" + "="*80)
print("common IDs:", len(common_ids))

for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(common_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)
    print(model_name, "rows after common ID filter:", len(datasets[model_name]))

# 代表としてGBSWからSMILESと目的変数を取る
base_df = datasets["GBSW"][[id_col, smiles_col, target_col]].copy()

# =========================
# 3. Fingerprint作成
# =========================

fps = []
valid_indices = []

for i, smi in enumerate(base_df[smiles_col]):
    arr = smiles_to_morgan_fp(smi, radius=FP_RADIUS, n_bits=FP_BITS)
    if arr is not None:
        fps.append(arr)
        valid_indices.append(i)

print("\n" + "="*80)
print("fingerprint")
print("valid molecules:", len(valid_indices))
print("invalid molecules:", len(base_df) - len(valid_indices))

base_df_valid = base_df.iloc[valid_indices].copy().reset_index(drop=True)

X_fp = np.vstack(fps).astype(float)
y = base_df_valid[target_col].values.astype(float)
ids = base_df_valid[id_col].values.astype(int)

valid_ids = set(ids)

print("X_fp shape:", X_fp.shape)
print("y shape   :", y.shape)

# 各MDデータもvalid_idsに合わせる
for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(valid_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)

    # ID順がfingerprint側と一致しているか確認
    assert np.array_equal(datasets[model_name][id_col].values, ids)

# =========================
# 4. 同一train/test split
# =========================

indices = np.arange(len(ids))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("\n===== split =====")
print("n train:", len(train_idx))
print("n test :", len(test_idx))

# =========================
# 5. Fingerprint only / Fingerprint + MD25 比較
# =========================

results = []
predictions = []

# Fingerprint only
res, pred, rf = evaluate_rf(
    X=X_fp,
    y=y,
    ids=ids,
    train_idx=train_idx,
    test_idx=test_idx,
    feature_set_name="Fingerprint only"
)
results.append(res)
predictions.append(pred)

# Fingerprint + each MD model
for model_name, df in datasets.items():
    X_md = df[md25_cols].values.astype(float)

    X_combo = np.hstack([X_fp, X_md])

    feature_set_name = f"Fingerprint + {model_name} MD25"

    res, pred, rf = evaluate_rf(
        X=X_combo,
        y=y,
        ids=ids,
        train_idx=train_idx,
        test_idx=test_idx,
        feature_set_name=feature_set_name
    )

    results.append(res)
    predictions.append(pred)

# =========================
# 6. 保存
# =========================

results_df = pd.DataFrame(results)
predictions_df = pd.concat(predictions, ignore_index=True)

metrics_out = os.path.join(save_dir, "rf_fingerprint_plus_3model_MD25_strictPAMPA_metrics.csv")
pred_out = os.path.join(save_dir, "rf_fingerprint_plus_3model_MD25_strictPAMPA_predictions.csv")

results_df.to_csv(metrics_out, index=False)
predictions_df.to_csv(pred_out, index=False)

print("\n" + "="*80)
print("Saved:")
print(metrics_out)
print(pred_out)

print("\n===== Fingerprint vs Fingerprint + MD25 =====")
display(results_df.sort_values("R2", ascending=False))


GBSW
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
PAMPA_logPm describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
max        -3.900000
Name: PAMPA_logPm, dtype: float64

HDGB
path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
PAMPA_logPm describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
max        -3.900000
Name: PAMPA_logPm, dtype: float64

HDGBvdW
path: /Users/teraimao/experiment/data/解析結果/hdgbvdw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
PAMPA_logPm describe:
count    6441.000000
mean       -5.733863
std         0.756894
min        -9.290000
25%        -6.140000
50%        -5.650000
75%        -5.230000
ma

[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerator
[16:40:15] DEPRECATION WARNING: please use MorganGenerat


fingerprint
valid molecules: 6441
invalid molecules: 0
X_fp shape: (6441, 2048)
y shape   : (6441,)

===== split =====
n train: 5152
n test : 1289

Fingerprint only
n features: 2048
n train   : 5152
n test    : 1289
Pearson R : 0.7858
R2        : 0.6048
MAE       : 0.3683
RMSE      : 0.4799

Fingerprint + GBSW MD25
n features: 2073
n train   : 5152
n test    : 1289
Pearson R : 0.7878
R2        : 0.6024
MAE       : 0.3695
RMSE      : 0.4814

Fingerprint + HDGB MD25
n features: 2073
n train   : 5152
n test    : 1289
Pearson R : 0.7892
R2        : 0.6038
MAE       : 0.3679
RMSE      : 0.4806

Fingerprint + HDGBvdW MD25
n features: 2073
n train   : 5152
n test    : 1289
Pearson R : 0.7931
R2        : 0.6093
MAE       : 0.3649
RMSE      : 0.4772

Saved:
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_plus_3model_MD25_strictPAMPA_metrics.csv
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_plus_3model_MD25_strictPAMPA_predictions.csv

===== Fingerprint vs Fingerprint + MD25 =====


,Feature_set,Dataset,N_total_common_ID,N_train,N_test,N_features,FP_radius,FP_bits,R,R2,MAE,RMSE
3,Fingerprint + HDGBvdW MD25,strict_PAMPA,6441,5152,1289,2073,2,2048,0.793111,0.609279,0.364915,0.477221
0,Fingerprint only,strict_PAMPA,6441,5152,1289,2048,2,2048,0.785792,0.604845,0.368293,0.479921
2,Fingerprint + HDGB MD25,strict_PAMPA,6441,5152,1289,2073,2,2048,0.789207,0.603764,0.367886,0.480577
1,Fingerprint + GBSW MD25,strict_PAMPA,6441,5152,1289,2073,2,2048,0.787756,0.602432,0.369474,0.481384


# 20回分の平均で比較

## MDのみ

In [18]:

# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "GBSW": "gbsw_common_successID_baselineB_strictPAMPA.csv",
    "HDGB": "hdgb_common_successID_baselineB_strictPAMPA.csv",
    "HDGBvdW": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
}

id_col = "ID"
target_col = "PAMPA_logPm"

base_md_features = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

feature_cols = [
    f"{feat}_pos{pos}"
    for feat in base_md_features
    for pos in [1, 2, 3, 4, 5]
]

TEST_SIZE = 0.2

# 20回評価
SEEDS = list(range(20))

# =========================
# データ読み込み
# =========================

datasets = {}

for model_name, fname in files.items():
    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path, low_memory=False)

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    df = df.dropna(subset=[id_col, target_col]).copy()
    df[id_col] = df[id_col].astype(int)

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{model_name}: 足りないMD特徴量があります: {missing_cols}")

    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=feature_cols).copy()
    df = df.drop_duplicates(subset=[id_col], keep="first").copy()
    df = df.sort_values(id_col).reset_index(drop=True)

    print("\n" + "="*80)
    print(model_name)
    print("path:", path)
    print("rows:", len(df))
    print("unique IDs:", df[id_col].nunique())
    print("target NaN:", df[target_col].isna().sum())
    print("MD25 NaN total:", df[feature_cols].isna().sum().sum())

    datasets[model_name] = df[[id_col, target_col] + feature_cols].copy()

# =========================
# 3モデル共通IDにそろえる
# =========================

common_ids = None

for model_name, df in datasets.items():
    ids = set(df[id_col].unique())
    common_ids = ids if common_ids is None else common_ids & ids

common_ids = sorted(common_ids)

print("\n" + "="*80)
print("common IDs:", len(common_ids))

for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(common_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)

    print(model_name, "rows after common ID filter:", len(datasets[model_name]))

# ID順が全モデルで一致しているか確認
base_ids = datasets["GBSW"][id_col].values
for model_name in datasets:
    assert np.array_equal(datasets[model_name][id_col].values, base_ids)

# =========================
# 評価関数
# =========================

def evaluate_one_model(df, model_name, seed):
    X = df[feature_cols].values.astype(float)
    y = df[target_col].values.astype(float)
    ids = df[id_col].values.astype(int)

    indices = np.arange(len(df))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=seed
    )

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=seed,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2,
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    r, p = pearsonr(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return {
        "MD_model": model_name,
        "seed": seed,
        "Feature_set": f"{model_name} MD25 only",
        "Dataset": "strict_PAMPA",
        "N_total_common_ID": len(df),
        "N_train": len(train_idx),
        "N_test": len(test_idx),
        "N_features": X.shape[1],
        "R": r,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
    }

# =========================
# 20 seeds × 3モデル
# =========================

all_results = []

for seed in SEEDS:
    print("\n" + "="*80)
    print("seed:", seed)

    for model_name, df in datasets.items():
        result = evaluate_one_model(df, model_name, seed)
        all_results.append(result)

        print(
            f"{model_name:7s} "
            f"R={result['R']:.4f} "
            f"R2={result['R2']:.4f} "
            f"MAE={result['MAE']:.4f} "
            f"RMSE={result['RMSE']:.4f}"
        )

results_df = pd.DataFrame(all_results)

# =========================
# 平均±標準偏差
# =========================

summary_df = (
    results_df
    .groupby(["MD_model", "Feature_set"], as_index=False)
    .agg(
        N_runs=("seed", "count"),
        N_total_common_ID=("N_total_common_ID", "first"),
        N_train=("N_train", "first"),
        N_test=("N_test", "first"),
        N_features=("N_features", "first"),

        R_mean=("R", "mean"),
        R_std=("R", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
    )
)

summary_df = summary_df.sort_values("R2_mean", ascending=False).reset_index(drop=True)

# 表示用
display_df = summary_df.copy()

for metric in ["R", "R2", "MAE", "RMSE"]:
    display_df[f"{metric}_mean±std"] = (
        display_df[f"{metric}_mean"].round(4).astype(str)
        + " ± "
        + display_df[f"{metric}_std"].round(4).astype(str)
    )

display_cols = [
    "MD_model",
    "N_runs",
    "N_total_common_ID",
    "N_train",
    "N_test",
    "N_features",
    "R_mean±std",
    "R2_mean±std",
    "MAE_mean±std",
    "RMSE_mean±std",
]

print("\n" + "="*80)
print("===== 20 seeds summary: each MD model only =====")
display(display_df[display_cols])

# =========================
# 保存
# =========================

eachrun_out = os.path.join(
    save_dir,
    "rf_3model_MD25_only_strictPAMPA_20seeds_eachrun.csv"
)

summary_out = os.path.join(
    save_dir,
    "rf_3model_MD25_only_strictPAMPA_20seeds_summary.csv"
)

results_df.to_csv(eachrun_out, index=False)
summary_df.to_csv(summary_out, index=False)

print("\nSaved:")
print(eachrun_out)
print(summary_out)


GBSW
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

HDGB
path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

HDGBvdW
path: /Users/teraimao/experiment/data/解析結果/hdgbvdw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

common IDs: 6441
GBSW rows after common ID filter: 6441
HDGB rows after common ID filter: 6441
HDGBvdW rows after common ID filter: 6441

seed: 0
GBSW    R=0.6551 R2=0.4186 MAE=0.4400 RMSE=0.5835
HDGB    R=0.6541 R2=0.4151 MAE=0.4436 RMSE=0.5853
HDGBvdW R=0.6481 R2=0.4118 MAE=0.4382 RMSE=0.5869

seed: 1
GBSW    R=0.6168 R2=0.3767 MAE=0.4389 RMSE=0.5739
HDGB    R=0.5996 R2=0.3562 MAE=0.4452 RMSE=0.5833
HDGBvdW R=0.6137 R2=0.3731 MAE=0.4369 RMSE=0.5756

seed: 2
GBSW    R=0.6551 R2=0.4185 MAE=0.4387 RMSE=0.5869
HDGB    

,MD_model,N_runs,N_total_common_ID,N_train,N_test,N_features,R_mean±std,R2_mean±std,MAE_mean±std,RMSE_mean±std
0,GBSW,20,6441,5152,1289,25,0.6435 ± 0.014,0.4078 ± 0.0163,0.4396 ± 0.0077,0.5799 ± 0.0111
1,HDGBvdW,20,6441,5152,1289,25,0.6363 ± 0.015,0.4 ± 0.0177,0.4388 ± 0.0088,0.5837 ± 0.0122
2,HDGB,20,6441,5152,1289,25,0.6308 ± 0.0164,0.3915 ± 0.0188,0.4454 ± 0.0079,0.5878 ± 0.0118



Saved:
/Users/teraimao/experiment/data/解析結果/rf_3model_MD25_only_strictPAMPA_20seeds_eachrun.csv
/Users/teraimao/experiment/data/解析結果/rf_3model_MD25_only_strictPAMPA_20seeds_summary.csv


## Fingerprint + MD特徴量

In [19]:

# =========================
# 設定
# =========================

save_dir = "/Users/teraimao/experiment/data/解析結果"

files = {
    "GBSW": "gbsw_common_successID_baselineB_strictPAMPA.csv",
    "HDGB": "hdgb_common_successID_baselineB_strictPAMPA.csv",
    "HDGBvdW": "hdgbvdw_common_successID_baselineB_strictPAMPA.csv",
}

id_col = "ID"
smiles_col = "SMILES"
target_col = "PAMPA_logPm"

base_md_features = [
    "HB_Avg",
    "SASA_Avg",
    "Solv_Free_Avg",
    "vdW_Avg",
    "Coul_Avg",
]

md25_cols = [
    f"{feat}_pos{pos}"
    for feat in base_md_features
    for pos in [1, 2, 3, 4, 5]
]

TEST_SIZE = 0.2
SEEDS = list(range(20))

FP_RADIUS = 2
FP_BITS = 2048

# =========================
# Fingerprint作成関数
# =========================

def smiles_to_morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr


def calc_metrics(y_true, y_pred):
    r, p = pearsonr(np.asarray(y_true), np.asarray(y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return r, r2, mae, rmse


def run_rf(X, y, seed):
    indices = np.arange(len(y))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=seed
    )

    X_train = X[train_idx]
    X_test = X[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    rf = RandomForestRegressor(
        n_estimators=500,
        random_state=seed,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2,
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    r, r2, mae, rmse = calc_metrics(y_test, y_pred)

    return {
        "seed": seed,
        "N_train": len(train_idx),
        "N_test": len(test_idx),
        "R": r,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
    }

# =========================
# 1. データ読み込み
# =========================

datasets = {}

for model_name, fname in files.items():
    path = os.path.join(save_dir, fname)
    df = pd.read_csv(path, low_memory=False)

    df[id_col] = pd.to_numeric(df[id_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    df = df.dropna(subset=[id_col, smiles_col, target_col]).copy()
    df[id_col] = df[id_col].astype(int)
    df = df.drop_duplicates(subset=[id_col], keep="first").copy()

    missing_md = [c for c in md25_cols if c not in df.columns]
    if missing_md:
        raise ValueError(f"{model_name}: MD25列が足りません: {missing_md}")

    for c in md25_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=md25_cols).copy()
    df = df.sort_values(id_col).reset_index(drop=True)

    print("\n" + "="*80)
    print(model_name)
    print("path:", path)
    print("rows:", len(df))
    print("unique IDs:", df[id_col].nunique())
    print("target NaN:", df[target_col].isna().sum())
    print("MD25 NaN total:", df[md25_cols].isna().sum().sum())

    datasets[model_name] = df[[id_col, smiles_col, target_col] + md25_cols].copy()

# =========================
# 2. 3モデル共通IDにそろえる
# =========================

common_ids = None

for model_name, df in datasets.items():
    ids = set(df[id_col].unique())
    common_ids = ids if common_ids is None else common_ids & ids

common_ids = sorted(common_ids)

print("\n" + "="*80)
print("common IDs:", len(common_ids))

for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(common_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)

    print(model_name, "rows after common ID filter:", len(datasets[model_name]))

# ID順確認
base_ids = datasets["GBSW"][id_col].values
for model_name in datasets:
    assert np.array_equal(datasets[model_name][id_col].values, base_ids)

# =========================
# 3. Fingerprint作成
# =========================

base_df = datasets["GBSW"][[id_col, smiles_col, target_col]].copy()

fps = []
valid_indices = []

for i, smi in enumerate(base_df[smiles_col]):
    arr = smiles_to_morgan_fp(smi, radius=FP_RADIUS, n_bits=FP_BITS)

    if arr is not None:
        fps.append(arr)
        valid_indices.append(i)

print("\n" + "="*80)
print("fingerprint")
print("valid molecules:", len(valid_indices))
print("invalid molecules:", len(base_df) - len(valid_indices))

base_df_valid = base_df.iloc[valid_indices].copy().reset_index(drop=True)

X_fp = np.vstack(fps).astype(float)
y = base_df_valid[target_col].values.astype(float)
ids = base_df_valid[id_col].values.astype(int)

valid_ids = set(ids)

print("X_fp shape:", X_fp.shape)
print("y shape   :", y.shape)

# MD側もfingerprint作成成功IDに合わせる
for model_name in datasets:
    datasets[model_name] = datasets[model_name][
        datasets[model_name][id_col].isin(valid_ids)
    ].copy()
    datasets[model_name] = datasets[model_name].sort_values(id_col).reset_index(drop=True)

    assert np.array_equal(datasets[model_name][id_col].values, ids)

# =========================
# 4. 20 seeds 評価
# =========================

all_results = []

for seed in SEEDS:
    print("\n" + "="*80)
    print("seed:", seed)

    # Fingerprint only
    res = run_rf(X_fp, y, seed)
    res.update({
        "Feature_set": "Fingerprint only",
        "MD_model": "None",
        "Dataset": "strict_PAMPA",
        "N_total_common_ID": len(y),
        "N_features": X_fp.shape[1],
        "FP_radius": FP_RADIUS,
        "FP_bits": FP_BITS,
    })
    all_results.append(res)

    print(
        f"{'Fingerprint only':30s} "
        f"R={res['R']:.4f} "
        f"R2={res['R2']:.4f} "
        f"MAE={res['MAE']:.4f} "
        f"RMSE={res['RMSE']:.4f}"
    )

    # Fingerprint + each MD25
    for model_name, df in datasets.items():
        X_md = df[md25_cols].values.astype(float)
        X_combo = np.hstack([X_fp, X_md])

        feature_set = f"Fingerprint + {model_name} MD25"

        res = run_rf(X_combo, y, seed)
        res.update({
            "Feature_set": feature_set,
            "MD_model": model_name,
            "Dataset": "strict_PAMPA",
            "N_total_common_ID": len(y),
            "N_features": X_combo.shape[1],
            "FP_radius": FP_RADIUS,
            "FP_bits": FP_BITS,
        })
        all_results.append(res)

        print(
            f"{feature_set:30s} "
            f"R={res['R']:.4f} "
            f"R2={res['R2']:.4f} "
            f"MAE={res['MAE']:.4f} "
            f"RMSE={res['RMSE']:.4f}"
        )

results_df = pd.DataFrame(all_results)

# =========================
# 5. 平均±標準偏差
# =========================

summary_df = (
    results_df
    .groupby(["Feature_set", "MD_model"], as_index=False)
    .agg(
        N_runs=("seed", "count"),
        N_total_common_ID=("N_total_common_ID", "first"),
        N_train=("N_train", "first"),
        N_test=("N_test", "first"),
        N_features=("N_features", "first"),
        FP_radius=("FP_radius", "first"),
        FP_bits=("FP_bits", "first"),

        R_mean=("R", "mean"),
        R_std=("R", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
    )
)

summary_df = summary_df.sort_values("R2_mean", ascending=False).reset_index(drop=True)

display_df = summary_df.copy()

for metric in ["R", "R2", "MAE", "RMSE"]:
    display_df[f"{metric}_mean±std"] = (
        display_df[f"{metric}_mean"].round(4).astype(str)
        + " ± "
        + display_df[f"{metric}_std"].round(4).astype(str)
    )

display_cols = [
    "Feature_set",
    "MD_model",
    "N_runs",
    "N_total_common_ID",
    "N_train",
    "N_test",
    "N_features",
    "R_mean±std",
    "R2_mean±std",
    "MAE_mean±std",
    "RMSE_mean±std",
]

print("\n" + "="*80)
print("===== 20 seeds summary: Fingerprint only / Fingerprint + each MD25 =====")
display(display_df[display_cols])

# =========================
# 6. Fingerprint onlyとの差分
# =========================

fp_row = summary_df[summary_df["Feature_set"] == "Fingerprint only"].iloc[0]

diff_df = summary_df.copy()
for metric in ["R", "R2", "MAE", "RMSE"]:
    diff_df[f"delta_{metric}_mean_vs_FP"] = diff_df[f"{metric}_mean"] - fp_row[f"{metric}_mean"]

print("\n" + "="*80)
print("===== Difference from Fingerprint only =====")
display(diff_df[
    [
        "Feature_set",
        "MD_model",
        "R2_mean",
        "delta_R2_mean_vs_FP",
        "MAE_mean",
        "delta_MAE_mean_vs_FP",
        "RMSE_mean",
        "delta_RMSE_mean_vs_FP",
    ]
].sort_values("delta_R2_mean_vs_FP", ascending=False))

# =========================
# 7. 保存
# =========================

eachrun_out = os.path.join(
    save_dir,
    "rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_eachrun.csv"
)

summary_out = os.path.join(
    save_dir,
    "rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_summary.csv"
)

diff_out = os.path.join(
    save_dir,
    "rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_diff_from_FP.csv"
)

results_df.to_csv(eachrun_out, index=False)
summary_df.to_csv(summary_out, index=False)
diff_df.to_csv(diff_out, index=False)

print("\nSaved:")
print(eachrun_out)
print(summary_out)
print(diff_out)


GBSW
path: /Users/teraimao/experiment/data/解析結果/gbsw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

HDGB
path: /Users/teraimao/experiment/data/解析結果/hdgb_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

HDGBvdW
path: /Users/teraimao/experiment/data/解析結果/hdgbvdw_common_successID_baselineB_strictPAMPA.csv
rows: 6441
unique IDs: 6441
target NaN: 0
MD25 NaN total: 0

common IDs: 6441
GBSW rows after common ID filter: 6441
HDGB rows after common ID filter: 6441
HDGBvdW rows after common ID filter: 6441


[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerator
[17:02:06] DEPRECATION WARNING: please use MorganGenerat


fingerprint
valid molecules: 6441
invalid molecules: 0
X_fp shape: (6441, 2048)
y shape   : (6441,)

seed: 0
Fingerprint only               R=0.7702 R2=0.5836 MAE=0.3699 RMSE=0.4938
Fingerprint + GBSW MD25        R=0.7690 R2=0.5774 MAE=0.3727 RMSE=0.4975
Fingerprint + HDGB MD25        R=0.7681 R2=0.5749 MAE=0.3747 RMSE=0.4989
Fingerprint + HDGBvdW MD25     R=0.7723 R2=0.5797 MAE=0.3731 RMSE=0.4961

seed: 1
Fingerprint only               R=0.7476 R2=0.5536 MAE=0.3652 RMSE=0.4857
Fingerprint + GBSW MD25        R=0.7483 R2=0.5504 MAE=0.3677 RMSE=0.4874
Fingerprint + HDGB MD25        R=0.7504 R2=0.5522 MAE=0.3676 RMSE=0.4865
Fingerprint + HDGBvdW MD25     R=0.7546 R2=0.5589 MAE=0.3638 RMSE=0.4828

seed: 2
Fingerprint only               R=0.7770 R2=0.5921 MAE=0.3683 RMSE=0.4915
Fingerprint + GBSW MD25        R=0.7814 R2=0.5913 MAE=0.3689 RMSE=0.4920
Fingerprint + HDGB MD25        R=0.7823 R2=0.5917 MAE=0.3705 RMSE=0.4917
Fingerprint + HDGBvdW MD25     R=0.7845 R2=0.5955 MAE=0.3664 RMSE=0.4

,Feature_set,MD_model,N_runs,N_total_common_ID,N_train,N_test,N_features,R_mean±std,R2_mean±std,MAE_mean±std,RMSE_mean±std
0,Fingerprint + HDGBvdW MD25,HDGBvdW,20,6441,5152,1289,2073,0.7663 ± 0.012,0.5754 ± 0.0158,0.3675 ± 0.0074,0.491 ± 0.0111
1,Fingerprint only,None,20,6441,5152,1289,2048,0.7621 ± 0.0111,0.5735 ± 0.0156,0.368 ± 0.0081,0.4921 ± 0.0117
2,Fingerprint + GBSW MD25,GBSW,20,6441,5152,1289,2073,0.7647 ± 0.0127,0.573 ± 0.0171,0.3698 ± 0.0075,0.4924 ± 0.0113
3,Fingerprint + HDGB MD25,HDGB,20,6441,5152,1289,2073,0.7639 ± 0.0122,0.5715 ± 0.0165,0.3699 ± 0.0074,0.4932 ± 0.0112



===== Difference from Fingerprint only =====


,Feature_set,MD_model,R2_mean,delta_R2_mean_vs_FP,MAE_mean,delta_MAE_mean_vs_FP,RMSE_mean,delta_RMSE_mean_vs_FP
0,Fingerprint + HDGBvdW MD25,HDGBvdW,0.575439,0.001914,0.367512,-0.000456,0.490976,-0.001126
1,Fingerprint only,None,0.573525,0.000000,0.367968,0.000000,0.492103,0.000000
2,Fingerprint + GBSW MD25,GBSW,0.572987,-0.000537,0.369758,0.001790,0.492373,0.000270
3,Fingerprint + HDGB MD25,HDGB,0.571543,-0.001981,0.369924,0.001956,0.493215,0.001113



Saved:
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_eachrun.csv
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_summary.csv
/Users/teraimao/experiment/data/解析結果/rf_fingerprint_plus_eachMD25_strictPAMPA_20seeds_diff_from_FP.csv


# 後ろ